<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [8]</a>'.</span>

# 10b_region_specific — Per-Region Differential Accessibility

**Goal:** Run evolutionary branch DESeq2 separately within each anatomical region
(Duodenum and Colon) to identify:
1. **Region-conserved evolutionary peaks** — same species contrast significant in both regions
2. **Region-specific evolutionary peaks** — contrast significant in only one region
3. **Comparison tables** per cell type and contrast showing overlap between regions

**Requires:** `pb_data_shared_per_region.rds` and `pb_data_union_per_region.rds`
from notebook `10a` (Cell 10).

**Key design:** Each pseudobulk sample = donor × cell_type × region (NOT aggregated).
The same orthology-aware peak subsetting and per-contrast outlier filtering are applied
as in `10b`. Only adult samples are used.

**Outputs:**
- `differential_results/region_specific/{Region}/{CellType}/{Contrast}.csv`
- `differential_results/region_specific/region_res_list.rds`
- `differential_results/region_specific/ultra_robust/`
- `differential_results/region_specific/comparison_tables/` — Duodenum vs Colon overlap
- `plots/region_specific/` — region comparison heatmaps + staircase plots per region

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(ggrepel)
  library(pheatmap)
  library(viridis)
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
  library(dplyr)
  library(tibble)
  library(readr)
})

source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

Packages loaded & utilities sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration & Load Per-Region Data
# =============================================================================
BASE    <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

TARGET_REGIONS <- c("Duodenum", "Colon")

save_dir <- file.path(OUT_DIR, "pseudobulk")

# Load region-preserved data (NOT region-aggregated, from 10a Cell 10)
pb_shared_reg <- readRDS(file.path(save_dir, "pb_data_shared_per_region.rds"))
pb_union_reg  <- readRDS(file.path(save_dir, "pb_data_union_per_region.rds"))

counts_shared_reg <- pb_shared_reg$counts
meta_reg          <- pb_shared_reg$meta
counts_union_reg  <- pb_union_reg$counts

# Load master annotation & build orthology index
anno_file   <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

ortho_mat <- build_orthology_index(master_anno, SPECIES)

message("Region-preserved shared peaks: ", nrow(counts_shared_reg), " x ", ncol(counts_shared_reg))
message("Region-preserved union peaks : ", nrow(counts_union_reg),  " x ", ncol(counts_union_reg))
message("Regions in data: ", paste(levels(meta_reg$region), collapse = ", "))
message("\nSamples per region x species:")
print(table(meta_reg$region, meta_reg$species))

Orthology index: 1142003 peaks x 6 species



Region-preserved shared peaks: 71663 x 519



Region-preserved union peaks : 1142003 x 519



Regions in data: Colon, Duodenum




Samples per region x species:



          
           Human Bonobo Chimpanzee Gorilla Macaque Marmoset
  Colon      108     22         32      42      45       43
  Duodenum    46     33         39      49      18       42


## Define Contrasts & Run Per-Region DESeq2

Uses the same 19 evolutionary contrasts as `10b`, run separately within Duodenum and Colon.
Per-contrast outlier filtering is active (2.5 SD threshold within species groups).

In [3]:
# =============================================================================
# Cell 3: Define Contrasts & Run Per-Region DESeq2
# =============================================================================
evo_contrasts <- define_evolutionary_contrasts()
message("Running per-region DESeq2 for ", length(evo_contrasts),
        " contrasts x ", length(TARGET_REGIONS), " regions")

# Skip expensive computation if checkpoint already exists (see Cell 3b)
rds_candidates_check <- c(
  file.path(OUT_DIR, "_summary", "region_res_list.rds"),
  file.path(OUT_DIR, "differential_results", "region_specific", "region_res_list.rds")
)
if (any(sapply(rds_candidates_check, file.exists))) {
  message("Checkpoint found — skipping DESeq2 re-run. Cell 3b will load from disk.")
  region_res <- NULL  # will be populated by Cell 3b
} else {
  region_res <- run_deseq2_per_region(
    counts_union_regions  = counts_union_reg,
    meta_regions          = meta_reg,
    contrasts             = evo_contrasts,
    ortho_mat             = ortho_mat,
    out_dir               = OUT_DIR,
    target_regions        = TARGET_REGIONS,
    min_samples           = 2,
    min_cells_region      = 50,
    min_counts_region     = 30000,
    filter_outliers       = TRUE,
    sd_thresh             = 2.5
  )
  message("\nPer-region DESeq2 complete.")
  message("Regions processed: ", paste(names(region_res), collapse = ", "))
}

Defined 19 evolutionary contrasts.



Running per-region DESeq2 for 19 contrasts x 2 regions



Checkpoint found <U+2014> skipping DESeq2 re-run. Cell 3b will load from disk.



In [4]:
# =============================================================================
# Cell 3b: Checkpoint — reload region_res if already saved
# =============================================================================
# Check new path first, fall back to old path from pre-migration runs
rds_candidates <- c(
  file.path(OUT_DIR, "_summary", "region_res_list.rds"),
  file.path(OUT_DIR, "differential_results", "region_specific", "region_res_list.rds")
)

if (!exists("region_res") || is.null(region_res)) {
  found <- Filter(file.exists, rds_candidates)
  if (length(found) > 0) {
    rds_path   <- found[[1]]
    region_res <- readRDS(rds_path)
    message("Loaded region_res from checkpoint: ", rds_path)
  } else {
    stop("region_res not found in memory or on disk — run Cell 3 first.")
  }
}
message("region_res regions: ", paste(names(region_res), collapse = ", "))

Loaded region_res from checkpoint: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/differential_results/region_specific/region_res_list.rds



region_res regions: Duodenum, Colon



## Ultra-Robust Filtering Per Region

Same `min(pos donors) > max(neg donors)` criterion, applied within each region's samples.

In [5]:
# =============================================================================
# Cell 4: Ultra-Robust Filtering Per Region
# =============================================================================
robust_peaks_per_region <- list()

for (region in TARGET_REGIONS) {
  message("\n--- Ultra-robust filtering: ", region, " ---")
  if (is.null(region_res[[region]])) next

  # Build a meta_shared-like object for this region to pass to ultra_robust_filter
  meta_this_region <- meta_reg[meta_reg$region == region, ]

  # ultra_robust_filter expects meta with cell_type and species
  robust_region <- ultra_robust_filter(
    branch_res   = region_res[[region]],
    counts_union = counts_union_reg,
    meta_shared  = meta_this_region,
    contrasts    = evo_contrasts,
    out_dir      = OUT_DIR,
    padj_thresh  = 0.05,
    lfc_thresh   = 1,
    min_logcpm   = 1
  )
  robust_peaks_per_region[[region]] <- robust_region

  # Robust count — sapply on an empty list returns list(), so use unlist + lapply
  n_total <- sum(unlist(lapply(robust_region, function(ct_data) {
    if (length(ct_data) == 0L) return(0L)
    sapply(ct_data, length)
  })))
  message("  Total ultra-robust peaks in ", region, ": ", n_total)
}

saveRDS(robust_peaks_per_region,
        file.path(OUT_DIR, "_summary",
                  "ultra_robust_per_region.rds"))
message("\nUltra-robust per-region results saved.")


--- Ultra-robust filtering: Duodenum ---




Ultra-robust filtering: BEST4..cells



  [Pair_Human_vs_Marmoset] DESeq2 UP: 8 <e2><86><92> Ultra-Robust: 8




Ultra-robust filtering: Crypt.Fibroblasts.WNT2B.



  [Node1_Human_vs_Pan] DESeq2 UP: 947 <e2><86><92> Ultra-Robust: 610



...

  [Pair_Human_vs_Macaque] DESeq2 UP: 1 <e2><86><92> Ultra-Robust: 1



  [Pair_Human_vs_Marmoset] DESeq2 UP: 191 <e2><86><92> Ultra-Robust: 191




Ultra-robust filtering complete. Saved checkpoint.



  Total ultra-robust peaks in Colon: 588767




Ultra-robust per-region results saved.



## Region Comparison: Duodenum vs Colon Overlap

For each cell type and contrast, classify significant peaks as:
- **Both**: significant in Duodenum AND Colon → region-conserved evolutionary peak
- **Duodenum_only / Colon_only**: significant in just one region → region-restricted

In [6]:
# =============================================================================
# Cell 5: Region Comparison Tables
# =============================================================================
comp_dir <- file.path(OUT_DIR, "_summary",
                       "comparison_tables")
dir.create(comp_dir, showWarnings = FALSE, recursive = TRUE)

# Collect all cell types and contrasts present in both regions
cell_types_both <- Reduce(intersect,
  lapply(TARGET_REGIONS, function(rg) names(region_res[[rg]])))

# Summary counts table: rows = contrast, cols = cell_type,
# values = n peaks sig in both, Duod_only, Colon_only
summary_rows <- list()

for (ct in cell_types_both) {
  contrasts_ct <- Reduce(intersect,
    lapply(TARGET_REGIONS, function(rg) names(region_res[[rg]][[ct]])))

  for (cn in contrasts_ct) {
    comp <- compare_regions(region_res, ct, cn, padj_thresh = 0.05, lfc_thresh = 1)
    if (is.null(comp) || nrow(comp) == 0) next

    # Save per cell_type × contrast comparison table
    out_f <- file.path(comp_dir, paste0(ct, "_", cn, "_region_comparison.csv"))
    write.csv(comp, out_f, row.names = FALSE)

    n_both  <- sum(comp$category == "Both")
    n_duod  <- sum(comp$category == "Duodenum_only")
    n_colon <- sum(comp$category == "Colon_only")

    summary_rows[[paste0(ct, "__", cn)]] <- data.frame(
      cell_type = ct, contrast = cn,
      n_both = n_both, n_duodenum_only = n_duod, n_colon_only = n_colon,
      n_total = n_both + n_duod + n_colon,
      pct_conserved = round(100 * n_both / max(1, n_both + n_duod + n_colon), 1),
      stringsAsFactors = FALSE
    )
  }
}

if (length(summary_rows) > 0) {
  summary_df <- do.call(rbind, summary_rows)
  write.csv(summary_df, file.path(comp_dir, "region_comparison_summary.csv"),
            row.names = FALSE)
  message("Region comparison summary saved: ", nrow(summary_df), " cell_type x contrast combinations")
  print(head(summary_df[order(-summary_df$pct_conserved), ], 20))
} else {
  message("No region comparison data generated.")
}

Region comparison summary saved: 164 cell_type x contrast combinations



                                                                                       cell_type
T.cells__ILS_HumanBonobo_vs_ChimpGorilla                                                 T.cells
T.cells__ILS_HumanGorilla_vs_Pan                                                         T.cells
T.cells__ILS_HumanChimp_vs_GorillaBonobo                                                 T.cells
Plasma.B.cells__ILS_HumanChimp_vs_GorillaBonobo                                   Plasma.B.cells
Plasma.B.cells__ILS_HumanGorilla_vs_Pan                                           Plasma.B.cells
Plasma.B.cells__ILS_HumanBonobo_vs_ChimpGorilla                                   Plasma.B.cells
T.cells__Div_Human_vs_Apes                                                               T.cells
T.cells__Div_Bonobo_vs_Apes                                                              T.cells
Goblet.cells__Pair_Human_vs_Chimp                                                   Goblet.cells
Macrophages__ILS_HumanGorilla_

## Region Comparison Heatmaps

For each cell type: a heatmap showing signed -log10(padj) across all region × contrast
combinations, for peaks significant in at least one combination.

In [7]:
# =============================================================================
# Cell 6: Region Comparison Heatmaps
# =============================================================================
plots_region_dir <- file.path(OUT_DIR, "plots", "region_specific")
dir.create(plots_region_dir, showWarnings = FALSE, recursive = TRUE)

for (ct in cell_types_both) {
  tryCatch({
    out_file <- file.path(plots_region_dir,
                          paste0("Region_Comparison_Heatmap_", ct, ".pdf"))
    plot_region_comparison_heatmap(region_res, ct, out_file)
  }, error = function(e) {
    message("  Heatmap failed for ", ct, ": ", e$message)
  })
}

message("\nRegion comparison heatmaps complete.")

  Heatmap failed for BEST4..cells: Viewport 'Signed
-log10(padj)_heatmap_body_1_1' was not found



  Subsampling 29605 <e2><86><92> 2000 peaks for region comparison heatmap



  Heatmap failed for Crypt.Fibroblasts.WNT2B.: Viewport 'Signed
-log10(padj)_heatmap_body_1_1' was not found



  Subsampling 5155 <e2><86><92> 2000 peaks for region comparison heatmap



  Heatmap failed for ECs: Viewport 'Signed
-log10(padj)_heatmap_body_1_1' was not found



...

  Subsampling 264625 <e2><86><92> 2000 peaks for region comparison heatmap



  Heatmap failed for TA.cells: Viewport 'Signed
-log10(padj)_heatmap_body_1_1' was not found



No significant peaks for region comparison: Tuft.cells



  Heatmap failed for Villus.Fibroblasts.WNT5B.: Viewport 'Signed
-log10(padj)_heatmap_body_1_1' was not found




Region comparison heatmaps complete.



## Per-Region Evolutionary Staircase Heatmaps

One staircase heatmap per region per cell type, showing ultra-robust peaks at each
phylogenetic node. Allows direct visual comparison of the evolutionary staircase
between Duodenum and Colon.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [8]:
# =============================================================================
# Cell 7: Per-Region Evolutionary Staircase Heatmaps
# =============================================================================
node_order <- c("Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
                "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
                "Node_Apes_vs_Monkeys", "Div_Human_vs_Apes",
                "Div_Human_vs_AllPrimates")

species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")

for (region in TARGET_REGIONS) {
  if (is.null(robust_peaks_per_region[[region]])) next
  robust_region <- robust_peaks_per_region[[region]]
  branch_region <- region_res[[region]]
  meta_this     <- meta_reg[meta_reg$region == region, ]

  for (ct in names(robust_region)) {
    ct_nodes <- intersect(node_order, names(robust_region[[ct]]))
    ct_nodes <- ct_nodes[sapply(ct_nodes,
      function(n) length(robust_region[[ct]][[n]]) > 0)]
    if (length(ct_nodes) == 0) next

    meta_ct    <- meta_this[meta_this$cell_type == ct, ]
    ct_samples <- intersect(rownames(meta_ct), colnames(counts_union_reg))
    ct_counts  <- counts_union_reg[, ct_samples, drop = FALSE]
    meta_ct    <- meta_ct[ct_samples, ]

    lib_sizes  <- colSums(ct_counts)
    cpm_mat    <- t(t(ct_counts) / lib_sizes) * 1e6
    logcpm_mat <- log2(cpm_mat + 1)

    avg_exp <- matrix(NA, nrow = nrow(logcpm_mat), ncol = length(species_order),
                      dimnames = list(rownames(logcpm_mat), species_order))
    for (sp in species_order) {
      sp_samp <- ct_samples[meta_ct$species == sp]
      if (length(sp_samp) > 1)
        avg_exp[, sp] <- rowMeans(logcpm_mat[, sp_samp, drop = FALSE])
      else if (length(sp_samp) == 1)
        avg_exp[, sp] <- logcpm_mat[, sp_samp]
    }

    plot_peaks  <- c()
    row_splits  <- c()
    for (node in ct_nodes) {
      ur_peaks <- robust_region[[ct]][[node]]
      if (length(ur_peaks) == 0) next
      res_df   <- as.data.frame(branch_region[[ct]][[node]])
      ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
      sorted   <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
      top      <- head(sorted, 50)
      plot_peaks <- c(plot_peaks, top)
      row_splits <- c(row_splits, rep(node, length(top)))
    }
    if (length(plot_peaks) < 5) next

    mat        <- avg_exp[plot_peaks, , drop = FALSE]
    valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
    mat        <- mat[, valid_cols, drop = FALSE]
    mat_scaled <- t(apply(mat, 1, scale))
    colnames(mat_scaled) <- colnames(mat)
    split_factor <- factor(row_splits, levels = ct_nodes)

    col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))
    ht <- Heatmap(mat_scaled, name = "Z-score", col = col_fun,
                  row_split = split_factor, row_title_rot = 0,
                  row_title_gp = gpar(fontsize = 9, fontface = "bold"),
                  row_gap = unit(2, "mm"), cluster_rows = TRUE,
                  cluster_columns = FALSE, show_row_names = FALSE,
                  show_column_names = TRUE, column_names_rot = 45,
                  column_title = paste("Evolutionary Staircase:", ct, "—", region),
                  column_title_gp = gpar(fontsize = 13, fontface = "bold"),
                  heatmap_legend_param = list(title = "Z-score"))

    ht_file <- file.path(plots_region_dir,
                          paste0("Staircase_", region, "_", ct, ".pdf"))
    dynamic_h <- max(8, length(plot_peaks) * 0.08 + length(ct_nodes))
    pdf(ht_file, width = 10, height = dynamic_h)
    draw(ht, merge_legend = TRUE)
    dev.off()
    message("  Staircase: ", region, " | ", ct, " (", length(plot_peaks), " peaks)")
  }
}

message("\nPer-region staircase heatmaps complete.")

ERROR: Error in ct_nodes[sapply(ct_nodes, function(n) length(robust_region[[ct]][[n]]) > : invalid subscript type 'list'


In [ ]:
# =============================================================================
# Cell 8: Summary Checkpoint
# =============================================================================
message("\n=== 10b_region_specific complete ===")
for (rg in TARGET_REGIONS) {
  if (is.null(region_res[[rg]])) next
  n_cts <- length(region_res[[rg]])
  n_contrasts <- sum(sapply(region_res[[rg]], length))
  message(sprintf("  %-12s: %d cell types, %d contrast results", rg, n_cts, n_contrasts))
}
message("\nOutputs: ",
  file.path(OUT_DIR, "_summary"))